In [ ]:


import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix, accuracy_score, precision_recall_fscore_support
from scipy.optimize import linear_sum_assignment

# ============================================================
# CONFIG
# ============================================================
TRAIN_DIR = "/kaggle/input/datasets/sabbir4724/training-data"  
VAL_DIR   = "/kaggle/input/datasets/sabbir4724/testing-data"     

NUM_CLUSTERS = 4          
PCA_DIM = 256             
IMAGE_SIZE = 224
BATCH_SIZE = 64
NUM_EPOCHS = 30          
LR = 0.05
MOMENTUM = 0.9
WEIGHT_DECAY = 1e-5
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)

# ============================================================
# DATA
# ============================================================
# Sobel-like normalization is commonly dropped in modern reimplementations;
# standard ImageNet normalization is used here (paper originally used
# grayscale + Sobel filtering, but RGB + ImageNet stats is the standard
# unmodified baseline used in most public DeepCluster reimplementations).
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])

# Deterministic transform used ONLY for feature extraction (no augmentation),
# as in the original DeepCluster feature-extraction pass.
feature_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])

train_dataset_aug = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
train_dataset_feat = datasets.ImageFolder(TRAIN_DIR, transform=feature_transform)
val_dataset = datasets.ImageFolder(VAL_DIR, transform=val_transform)

feature_loader = DataLoader(train_dataset_feat, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=True)

print(f"Train images: {len(train_dataset_aug)} | Val images: {len(val_dataset)}")
print(f"Val classes (for evaluation only): {val_dataset.classes}")


# ============================================================
# MODEL: standard DeepCluster convnet (AlexNet-style feature extractor
# as in the original paper) + linear classification head over clusters.
# ============================================================
class DeepClusterNet(nn.Module):
    """AlexNet-based feature extractor, matching the original DeepCluster
    paper's backbone (5 conv layers + fc6/fc7 as embedding), with a final
    classification head over the current number of pseudo-classes."""

    def __init__(self, num_clusters):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 96, kernel_size=11, stride=4, padding=2),
            nn.ReLU(inplace=True),
            nn.LocalResponseNorm(5, alpha=1e-4, beta=0.75, k=2),
            nn.MaxPool2d(kernel_size=3, stride=2),

            nn.Conv2d(96, 256, kernel_size=5, padding=2, groups=2),
            nn.ReLU(inplace=True),
            nn.LocalResponseNorm(5, alpha=1e-4, beta=0.75, k=2),
            nn.MaxPool2d(kernel_size=3, stride=2),

            nn.Conv2d(256, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(384, 384, kernel_size=3, padding=1, groups=2),
            nn.ReLU(inplace=True),

            nn.Conv2d(384, 256, kernel_size=3, padding=1, groups=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )
        self.avgpool = nn.AdaptiveAvgPool2d((6, 6))
        self.embedding = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
        )
        self.classifier = nn.Linear(4096, num_clusters)

    def forward(self, x, return_features=False):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        feat = self.embedding(x)
        if return_features:
            return feat
        out = self.classifier(feat)
        return out

    def reset_classifier(self, num_clusters):
        self.classifier = nn.Linear(4096, num_clusters).to(DEVICE)


model = DeepClusterNet(NUM_CLUSTERS).to(DEVICE)


# ============================================================
# FEATURE EXTRACTION
# ============================================================
@torch.no_grad()
def extract_features(net, loader):
    net.eval()
    feats = []
    for images, _ in loader:
        images = images.to(DEVICE)
        f = net(images, return_features=True)
        feats.append(f.cpu().numpy())
    return np.concatenate(feats, axis=0)


# ============================================================
# CLUSTERING: PCA-whitening + L2-norm + k-means
# (exactly as in the original DeepCluster pipeline)
# ============================================================
def cluster_features(features, num_clusters):
    pca_dim = min(PCA_DIM, features.shape[1], features.shape[0])
    pca = PCA(n_components=pca_dim, whiten=True, random_state=SEED)
    reduced = pca.fit_transform(features)
    norms = np.linalg.norm(reduced, axis=1, keepdims=True)
    norms[norms == 0] = 1
    reduced = reduced / norms

    km = KMeans(n_clusters=num_clusters, n_init=20, random_state=SEED)
    pseudo_labels = km.fit_predict(reduced)
    return pseudo_labels


# ============================================================
# TRAINING (alternating: cluster -> assign pseudo-labels -> train one epoch)
# ============================================================
def train_one_epoch(net, dataset_aug, pseudo_labels, optimizer, criterion):
    net.train()
    dataset_aug.samples = [(p, int(pseudo_labels[i]))
                            for i, (p, _) in enumerate(dataset_aug.samples)]
    dataset_aug.targets = [s[1] for s in dataset_aug.samples]

    loader = DataLoader(dataset_aug, batch_size=BATCH_SIZE, shuffle=True,
                         num_workers=2, pin_memory=True, drop_last=True)

    total_loss, total_correct, total_n = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = net(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        total_correct += (outputs.argmax(1) == labels).sum().item()
        total_n += images.size(0)

    return total_loss / total_n, total_correct / total_n


# ============================================================
# EVALUATION on the separate validation dataset
# ============================================================
def hungarian_match(pred_clusters, true_labels, num_clusters, num_classes):
    size = max(num_clusters, num_classes)
    cm = np.zeros((size, size), dtype=np.int64)
    for p, t in zip(pred_clusters, true_labels):
        cm[p, t] += 1
    row_ind, col_ind = linear_sum_assignment(-cm)
    mapping = {r: c for r, c in zip(row_ind, col_ind)}
    mapped_preds = np.array([mapping[p] for p in pred_clusters])
    return mapped_preds


def evaluate_on_val(net):
    val_features = extract_features(net, val_loader)
    val_true_labels = np.array(val_dataset.targets)
    num_classes = len(val_dataset.classes)

    val_clusters = cluster_features(val_features, NUM_CLUSTERS)
    mapped_preds = hungarian_match(val_clusters, val_true_labels, NUM_CLUSTERS, num_classes)

    acc = accuracy_score(val_true_labels, mapped_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        val_true_labels, mapped_preds, average="macro", zero_division=0
    )
    return acc, precision, recall, f1


# ============================================================
# MAIN LOOP
# ============================================================
def main():
    optimizer = optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM,
                           weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(1, NUM_EPOCHS + 1):
        t0 = time.time()

        # Step 1: extract features on train set
        features = extract_features(model, feature_loader)

        # Step 2: cluster features -> pseudo-labels
        pseudo_labels = cluster_features(features, NUM_CLUSTERS)

        # Step 3: reset classification head (new cluster assignment each epoch)
        model.reset_classifier(NUM_CLUSTERS)
        optimizer = optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM,
                               weight_decay=WEIGHT_DECAY)

        # Step 4: train one epoch with pseudo-labels
        train_loss, train_acc = train_one_epoch(model, train_dataset_aug,
                                                  pseudo_labels, optimizer, criterion)

        dt = time.time() - t0
        print(f"Epoch [{epoch}/{NUM_EPOCHS}] "
              f"train_loss={train_loss:.4f} pseudo_acc={train_acc:.4f} time={dt:.1f}s")

    # Final evaluation on the separate validation dataset
    acc, precision, recall, f1 = evaluate_on_val(model)
    print("\n=== Validation Results (Hungarian-matched clusters vs true labels) ===")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")

    torch.save(model.state_dict(), "deepcluster_model.pth")
    print("\nModel saved to deepcluster_model.pth")


if __name__ == "__main__":
    main()

In [ ]:


import time
import torch

NUM_CLUSTERS = 4          
IMAGE_SIZE = 224
MODEL_PATH = "deepcluster_model.pth"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

WARMUP_RUNS = 20
TIMED_RUNS = 100


# ============================================================
# 1) Params (M)
# ============================================================
def count_params(net):
    total_params = sum(p.numel() for p in net.parameters())
    trainable_params = sum(p.numel() for p in net.parameters() if p.requires_grad)
    return total_params / 1e6, trainable_params / 1e6


# ============================================================
# 2) Time/Image (ms) and FPS
# ============================================================
@torch.no_grad()
def benchmark_speed(net, image_size, warmup=WARMUP_RUNS, runs=TIMED_RUNS):
    net.eval()
    dummy = torch.randn(1, 3, image_size, image_size, device=DEVICE)

    # warmup (important for GPU — first calls include kernel init/caching)
    for _ in range(warmup):
        _ = net(dummy)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    # timed runs
    start = time.perf_counter()
    for _ in range(runs):
        _ = net(dummy)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    end = time.perf_counter()

    total_time_s = end - start
    time_per_image_ms = (total_time_s / runs) * 1000
    fps = runs / total_time_s
    return time_per_image_ms, fps


def main():
    # DeepClusterNet is already defined earlier in this same notebook/kernel
    # (from the deepcluster_raw training code) — no import needed.
    model = DeepClusterNet(NUM_CLUSTERS).to(DEVICE)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))

    total_m, trainable_m = count_params(model)
    time_ms, fps = benchmark_speed(model, IMAGE_SIZE)

    print("=== Model Efficiency Metrics ===")
    print(f"Device          : {DEVICE}")
    print(f"Params (M)      : {total_m:.2f}  (trainable: {trainable_m:.2f})")
    print(f"Time/Image (ms) : {time_ms:.2f}")
    print(f"FPS             : {fps:.2f}")


if __name__ == "__main__":
    main()